In [2]:
import pandas as pd

books = pd.read_csv("../books_with_categories.csv")

In [3]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = -1)
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528688922524452},
  {'label': 'neutral', 'score': 0.005764589179307222},
  {'label': 'anger', 'score': 0.004419790115207434},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.0004138524236623198}]]

In [4]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [5]:
classifier(books["description"][0])

[[{'label': 'fear', 'score': 0.6548401713371277},
  {'label': 'neutral', 'score': 0.16985252499580383},
  {'label': 'sadness', 'score': 0.11640933156013489},
  {'label': 'surprise', 'score': 0.020700689405202866},
  {'label': 'disgust', 'score': 0.01910070702433586},
  {'label': 'joy', 'score': 0.015161466784775257},
  {'label': 'anger', 'score': 0.0039351508021354675}]]

In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(books["description"][0])

sentences = [sent.text for sent in doc.sents]
predictions = classifier(sentences)

In [8]:
sentences[0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives.'

In [9]:

predictions[0]

[{'label': 'surprise', 'score': 0.7214332818984985},
 {'label': 'neutral', 'score': 0.16815747320652008},
 {'label': 'fear', 'score': 0.05271964520215988},
 {'label': 'joy', 'score': 0.04348466917872429},
 {'label': 'anger', 'score': 0.009235912002623081},
 {'label': 'disgust', 'score': 0.0028505343943834305},
 {'label': 'sadness', 'score': 0.002118480857461691}]

In [10]:
sentences[3]

'Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist.'

In [11]:
predictions[3]

[{'label': 'fear', 'score': 0.9790166020393372},
 {'label': 'neutral', 'score': 0.006491743493825197},
 {'label': 'sadness', 'score': 0.00495960982516408},
 {'label': 'anger', 'score': 0.003323897486552596},
 {'label': 'surprise', 'score': 0.002782263793051243},
 {'label': 'disgust', 'score': 0.00276524038054049},
 {'label': 'joy', 'score': 0.0006605995586141944}]

In [12]:
for sent, pred in zip(sentences, predictions):
    print("Sentence:", sent)
    print("Emotion:", pred)
    print("------")

Sentence: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives.
Emotion: [{'label': 'surprise', 'score': 0.7214332818984985}, {'label': 'neutral', 'score': 0.16815747320652008}, {'label': 'fear', 'score': 0.05271964520215988}, {'label': 'joy', 'score': 0.04348466917872429}, {'label': 'anger', 'score': 0.009235912002623081}, {'label': 'disgust', 'score': 0.0028505343943834305}, {'label': 'sadness', 'score': 0.002118480857461691}]
------
Sentence: John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers.
Emotion: [{'label': 'disgust', 'score': 0.45429494976997375}, {'label': 'neutral', 'score': 0.44546207785606384}, {'label': 'joy', 'score': 0.0421014279127121}, {'label': 'sadness', 'score': 0.030259815976023674}, {'label': 'anger', 'score': 0.01675546169281006}, {'label': 'surprise', 'score': 0.007567726541310549}, {'label': 'fear', 'score': 0.0035

In [13]:
sorted(predictions[0], key=lambda x: x["label"])

[{'label': 'anger', 'score': 0.009235912002623081},
 {'label': 'disgust', 'score': 0.0028505343943834305},
 {'label': 'fear', 'score': 0.05271964520215988},
 {'label': 'joy', 'score': 0.04348466917872429},
 {'label': 'neutral', 'score': 0.16815747320652008},
 {'label': 'sadness', 'score': 0.002118480857461691},
 {'label': 'surprise', 'score': 0.7214332818984985}]

In [14]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [15]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [16]:
emotion_scores

{'anger': [np.float64(0.0641336664557457),
  np.float64(0.6126194596290588),
  np.float64(0.0641336664557457),
  np.float64(0.351485013961792),
  np.float64(0.08141247928142548),
  np.float64(0.23222415149211884),
  np.float64(0.5381836295127869),
  np.float64(0.0641336664557457),
  np.float64(0.3006701171398163),
  np.float64(0.0641336664557457)],
 'disgust': [np.float64(0.2735915184020996),
  np.float64(0.3482842445373535),
  np.float64(0.10400674492120743),
  np.float64(0.15072223544120789),
  np.float64(0.18449555337429047),
  np.float64(0.7271754741668701),
  np.float64(0.1558547466993332),
  np.float64(0.10400674492120743),
  np.float64(0.2794811725616455),
  np.float64(0.1779259592294693)],
 'fear': [np.float64(0.9281681180000305),
  np.float64(0.9425276517868042),
  np.float64(0.9723207950592041),
  np.float64(0.3607054352760315),
  np.float64(0.09504346549510956),
  np.float64(0.051362838596105576),
  np.float64(0.7474274635314941),
  np.float64(0.40449732542037964),
  np.floa

In [17]:
from tqdm import tqdm
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

# Pre-allocate (faster than append)
n = len(books)
isbn = books["isbn13"].tolist()
emotion_scores = {label: np.zeros(n) for label in emotion_labels}

batch_size = 32  # you can try 16, 32, 64 depending on VRAM

for start in tqdm(range(0, n, batch_size)):
    end = min(start + batch_size, n)

    # Get batch descriptions
    batch_desc = books["description"][start:end].tolist()

    # Split into sentences (flatten)
    all_sentences = []
    mapping = []  # track which book each sentence belongs to

    for idx, desc in enumerate(batch_desc):
        sentences = desc.split(".")
        all_sentences.extend(sentences)
        mapping.extend([idx] * len(sentences))

    # Run classifier ONCE per batch (this is the key speedup)
    predictions = classifier(all_sentences, batch_size=32, truncation=True)

    # Aggregate max scores per book
    temp_scores = [{label: 0 for label in emotion_labels} for _ in range(len(batch_desc))]

    for pred, book_idx in zip(predictions, mapping):
        for item in pred:
            label = item["label"].lower()
            score = item["score"]
            if score > temp_scores[book_idx][label]:
                temp_scores[book_idx][label] = score

    # Store results
    for i, scores in enumerate(temp_scores):
        for label in emotion_labels:
            emotion_scores[label][start + i] = scores[label]

100%|██████████| 163/163 [07:39<00:00,  2.82s/it]


In [18]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [19]:

emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.064134,0.273592,0.928168,0.932798,0.967158,0.729602,0.646216,9780002005883
1,0.612620,0.348285,0.942528,0.704422,0.111690,0.252547,0.887940,9780002261982
2,0.064134,0.104007,0.972321,0.767238,0.111690,0.078766,0.549477,9780006178736
3,0.351484,0.150722,0.360706,0.251881,0.111690,0.078766,0.732685,9780006280897
4,0.081413,0.184495,0.095043,0.040564,0.475880,0.078766,0.884390,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.148208,0.030643,0.919165,0.255172,0.980877,0.030656,0.853721,9788172235222
5193,0.064134,0.114383,0.051363,0.400263,0.111690,0.227765,0.883198,9788173031014
5194,0.009997,0.009929,0.339218,0.947779,0.066685,0.057625,0.375754,9788179921623
5195,0.064134,0.104007,0.459269,0.759456,0.368110,0.078765,0.951104,9788185300535


In [20]:
books = pd.merge(books, emotions_df, on = "isbn13")

In [21]:

books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273592,0.928168,0.932798,0.967158,0.729602,0.646216
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612620,0.348285,0.942528,0.704422,0.111690,0.252547,0.887940
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767238,0.111690,0.078766,0.549477
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351484,0.150722,0.360706,0.251881,0.111690,0.078766,0.732685
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081413,0.184495,0.095043,0.040564,0.475880,0.078766,0.884390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,...,Mistaken Identity,9788172235222 On A Train Journey Home To North...,Fiction,0.148208,0.030643,0.919165,0.255172,0.980877,0.030656,0.853721
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.064134,0.114383,0.051363,0.400263,0.111690,0.227765,0.883198
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.009997,0.009929,0.339218,0.947779,0.066685,0.057625,0.375754
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.064134,0.104007,0.459269,0.759456,0.368110,0.078765,0.951104


In [22]:
books.to_csv("books_with_emotions.csv", index = False)